# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [6]:
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=DWH_Lib;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)


# Đọc data từ SQL Server

In [12]:
df_ctdt = pd.read_csv("./data_chuongtrinhdaotao.csv")
print(df_ctdt)

    ID_CTDT                                       TenCTDT    KyHieu  NamBH  \
0         1                             Sư phạm tiếng Anh   7140231   2023   
1         2                               Thiết kế đồ họa   7210403   2023   
2         3                           Thiết kế thời trang   7210404   2023   
3         4                                  Ngôn ngữ Anh   7220201   2023   
4         5                            Kinh doanh Quốc tế   7340120   2023   
5         6                            Thương mại điện tử   7340122   2023   
6         7                                       Kế toán   7340301   2023   
7         8                                          Luật   7380101   2023   
8         9                   Công nghệ kỹ thuật máy tính   7480108   2023   
9        10                         Hệ thống nhúng và IoT   7480118   2023   
10       11                           Công nghệ thông tin   7480201   2023   
11       12                             An toàn thông tin   7480

# Xử lý data

In [13]:
new_row = pd.DataFrame({
    'KyHieu': ['0'],
    'TenCTDT': ['(Không xác định)'],
    'NamBH': [0],
    'ID_Khoa': [0]
})
df_ctdt = pd.concat([df_ctdt, new_row], ignore_index=True) # Thêm vào dataset
df_ctdt = df_ctdt.sort_values(by='KyHieu', ascending=True).reset_index(drop=True) # sắp xếp lại cho dễ nhìn
df_ctdt = df_ctdt.drop(columns=["ID_CTDT"])
print(df_ctdt)

                                         TenCTDT    KyHieu  NamBH  ID_Khoa
0                               (Không xác định)         0      0        0
1                              Sư phạm tiếng Anh   7140231   2023       12
2                              Sư phạm công nghệ  7140246V   2023       14
3                                Thiết kế đồ họa   7210403   2023        8
4                            Thiết kế thời trang   7210404   2023        4
5                                   Ngôn ngữ Anh   7220201   2023       12
6                            Tâm lý học giáo dục  7310403V   2023       14
7                             Kinh doanh Quốc tế   7340120   2023       10
8                             Thương mại điện tử   7340122   2023       10
9                                        Kế toán   7340301   2023       10
10                                          Luật   7380101   2023       11
11                   Công nghệ kỹ thuật máy tính   7480108   2023        7
12                       

# Load data

### [Nếu cần] Clear bảng

In [14]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_CTDT"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [15]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.DIM_CTDT (ID_ctdt, Ten_chuong_trinh_dao_tao, NamBH, ID_khoa) 
                VALUES (?, ?, ?, ?)
               """
for index, row in df_data_ctdt.iterrows():
    values = (row['KyHieu'], 
                row['TenCTDT'],
                row['NamBH'],
                row['ID_Khoa'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()